# 02 · Fine-tuning QLoRA (Qwen3-VL-8B-Instruct trên T4 15GB)
Giai đoạn 2: chia dataset 80/10/10 → fine-tune QLoRA 4-bit trên **Qwen3-VL-8B** (tải từ HuggingFace về local).

**Checkpoint adapter lưu thẳng vào Google Drive** → reconnect là train tiếp được.

> Runtime → Change runtime type → **GPU (T4)** trước khi chạy.
> Qwen3-VL cần `transformers` cài từ **source**.

## 1. Mount Drive + đường dẫn dataset, checkpoint, model local

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project')
REVIEWED    = PROJECT_DRIVE / 'data' / 'draft' / 'raw_draft.jsonl'  # đã duyệt 100%
DATASET_DIR = PROJECT_DRIVE / 'data' / 'dataset'                   # train/val/test
AUG_DIR     = PROJECT_DRIVE / 'data' / 'aug'
CKPT_DIR    = PROJECT_DRIVE / 'checkpoints' / 'qwen3vl-cccd-lora'   # adapter ra Drive
MODELS_DIR  = PROJECT_DRIVE / 'models'
QWEN3_LOCAL = MODELS_DIR / 'Qwen3-VL-8B-Instruct'
for d in (DATASET_DIR, AUG_DIR, CKPT_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Dataset   :', DATASET_DIR)
print('Checkpoint:', CKPT_DIR)
print('Model     :', QWEN3_LOCAL)

## 2. Cài thư viện QLoRA (transformers từ source cho Qwen3-VL)

In [ ]:
!cp -r '/content/drive/MyDrive/cccd_project/code' /content/cccd 2>/dev/null || echo 'Đặt source vào /content/cccd'
%cd /content/cccd
!pip -q install git+https://github.com/huggingface/transformers
!pip -q install qwen-vl-utils accelerate peft bitsandbytes Pillow tqdm huggingface_hub
!nvidia-smi

## 3. Tải Qwen3-VL-8B-Instruct từ HuggingFace về LOCAL (Drive)

In [ ]:
from huggingface_hub import snapshot_download
if not (QWEN3_LOCAL / 'config.json').exists():
    snapshot_download(repo_id='Qwen/Qwen3-VL-8B-Instruct',
                      local_dir=str(QWEN3_LOCAL),
                      ignore_patterns=['*.pth', 'original/*'])
    print('✓ Đã tải về', QWEN3_LOCAL)
else:
    print('✓ Model đã có sẵn local:', QWEN3_LOCAL)

## 4. Chia dataset 80/10/10 (augment CHỈ train, chống leakage)

In [ ]:
!python -m src.data_pipeline.prepare_dataset \
    --input '{REVIEWED}' \
    --out_dir '{DATASET_DIR}' \
    --aug_dir '{AUG_DIR}' \
    --image_root '{PROJECT_DRIVE}' \
    --n_aug 2 --ratios 0.8,0.1,0.1 --seed 42

## 5. Fine-tune QLoRA (4-bit NF4 + gradient checkpointing) — base LOCAL
batch_size=1 + grad_accum=8 → effective batch 8, vừa T4 15GB.

In [ ]:
!python scripts/train.py \
    --train_jsonl '{DATASET_DIR}/train.jsonl' \
    --val_jsonl   '{DATASET_DIR}/val.jsonl' \
    --image_root  '{PROJECT_DRIVE}' \
    --output_dir  '{CKPT_DIR}' \
    --model_name  '{QWEN3_LOCAL}' \
    --epochs 3 --batch_size 1 --grad_accum 8 --lr 1e-4 \
    --compute_dtype float16 --lora_r 16 --lora_alpha 32